# 🚗 TOYOTA SmartLife × EV LifePilot（合併版）
把「油車電車（5 年 TCO 精算 + CSV 統計問答）」與「SmartLife（車主檔案 + AI 對話）」合成一支程式。

**使用方式**
1. 把兩份 CSV（主檔檔名含 `CDP`、補充檔檔名含 `補充`）放進 Google 雲端硬碟的 `我的雲端硬碟/toyota/` 資料夾，程式會自動掛載雲端硬碟去找，之後每次開新的 Colab 都不用重傳；也可以直接上傳到 Colab 左側「檔案」，或在【區塊 0】填入路徑。
2. 左側「密鑰」新增名為 `api` 的 secret（放 Gemini API Key），並開啟「筆記本存取權」。
3. 由上到下依序執行所有儲存格，最後一格會產生 Gradio 網址。
4. **新增 / 抽換電動車款**：改【區塊 0】的 `EV_SPECS` 與 `DEFAULT_EV`；或不改程式，在 CSV 同一個資料夾放 `ev_catalog.json`（同名車款覆寫、新車款新增、`"enabled": false` 下架）。有專屬補充檔的車款填 `csv_keywords`，沒有的會直接用主檔資料。

In [1]:
!pip install -q -U google-genai
!pip install -q gradio pandas plotly scikit-learn

import os, re, glob, json, inspect
from dataclasses import dataclass, fields, replace as dc_replace
import numpy as np
import pandas as pd
import gradio as gr
import plotly.graph_objects as go
from google import genai
from google.genai import types
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 793.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 15.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.


In [2]:
#=========================================================
# 【區塊 0：參數設定】
# CSV 裡「沒有」的欄位（油價、電價、油耗、電耗）集中放這裡，之後只需要改這一區
# =========================================================
GAS_PRICE_PER_L    = 31.0   # 汽油 NT$/公升
ELEC_PRICE_PER_KWH = 4.5    # 電價 NT$/度
FUEL_KM_PER_L = {"汽油": 12.0, "油電混合": 20.0}  # 各動力型式平均油耗 km/L。汽油沿用原程式；油電混合是估計值，請依官方規格調整
FUEL_KM_PER_L_BY_MODEL = {}  # 想精準到單一車款時在這裡覆寫，例如 {"ALTIS": 14.5, "RAV4 HV": 21.0}

A_CURRENCY_TO_NTD = 1.0     # CSV 金額是「A國幣」(非台幣)。填 1 A國幣 = 幾元台幣；目前先以 1:1 當 NT$ 使用
YEARS = 5                   # TCO 計算年數
MAINT_AGE_BAND = (1, 5)     # 保養費只取「車齡 1~5 年」的車主，口徑與 5 年 TCO 一致（避免老車/新車混在一起比）
MIN_SAMPLE = 30             # 某車款在上述車齡區間內少於此人數 → 改用「同動力型式（汽油/油電混合/純電）車款」的平均

MAIN_CSV_PATH = None        # CSV 不在目前資料夾時，填完整路徑；None = 自動搜尋
SUPP_CSV_PATH = None        # bZ4X 補充檔路徑（已接到下方 bZ4X 的 EVSpec.csv_path；其他車款各自填 csv_path）

# 客群分群（KMeans）：依車齡 / 回廠次數把車主分成 高頻、低頻老車、標準 三群，再挑對應的話術方向
SEG_K = 3                   # 話術只定義了 3 種客群，請維持 3
SEG_MAX_ROWS = 300_000      # 分群訓練最多取幾筆（隨機抽樣；全量約 100 萬筆也可以，但會比較慢）
OUTSIDE_SHOP_KM = 20000     # 年里程超過這個值、且近一年回廠次數 <= OUTSIDE_SHOP_VISITS → 判斷「可能在外廠保養」
OUTSIDE_SHOP_VISITS = 1

# 充電情境（CSV 沒有這些資料，全部是「假設值」，請依原廠規格 / 實際電價調整；電池容量、電耗改到下方 EV 車款目錄，每台車各自設定）
CHARGE_WINDOW = (0.2, 0.8)            # 日常充電習慣：電量 20% → 80% 算一次充電
PUBLIC_CHARGE_PRICE_PER_KWH = 10.0    # 公共充電 NT$/度（假設值）
HOME_SHARE = {True: 0.9, False: 0.0}  # 有固定車位 → 九成電量在家充；沒有 → 全靠公共充電

# 長途情境（假設值）：高速行駛、冷氣、載重會讓實際電耗比日常差，所以續航打折
HIGHWAY_RANGE_FACTOR = 0.85           # 長途行駛時，實際 km/kWh 約為日常的幾成
TRIP_START_SOC, TRIP_ARRIVE_SOC, TRIP_CHARGE_TO = 0.9, 0.1, 0.8   # 出發電量 90%、抵達至少保留 10%、中途補電充到 80%

# =========================================================
# 【電動車款目錄（EV Catalog）】新增 / 抽換電動車款只需要改這裡，其餘區塊都會自動套用
#   - 新增車款：在 EV_SPECS 多加一個 EVSpec(...)
#   - 抽換車款：把舊車款設 enabled=False（或刪掉），再把 DEFAULT_EV 改成新車款
#   - 不想改程式：在 CSV 同一個資料夾放 ev_catalog.json，區塊 1 會讀它來「覆寫 / 新增 / 下架」車款
#   - 車款有專屬補充檔就填 csv_keywords；兩台以上各有補充檔時，關鍵字請加上車名，例如 ("補充", "EV_X")
# =========================================================
@dataclass(frozen=True)
class EVSpec:
    model: str                            # 必須和 CSV「現保有車款」欄位的寫法完全一樣（大小寫、空白都要一致）
    display: str                          # 介面、圖表上顯示的名稱
    km_per_kwh: float                     # 日常每度電可跑公里數（假設值，請依原廠規格調整）
    battery_kwh: float                    # 電池容量 kWh（假設值，請向原廠確認）
    csv_keywords: tuple = ()              # 專屬補充檔的檔名關鍵字；() = 直接用主檔裡這個車款的資料
    csv_path: str | None = None           # 補充檔完整路徑；填了就不自動搜尋
    maint_per_year: float | None = None   # CSV 樣本不足 / 新上市還沒車主資料時的年定保費假設值；None = 改用其他電動車款平均
    warranty: str = ""                    # 話術用的電池保固說法（不在 CSV 裡，請先向原廠確認）
    lifestyle: str = ""                   # 話術用的生活情境賣點（不在 CSV 裡，請先向原廠確認）
    enabled: bool = True                  # False = 暫時下架，不出現在試算與 AI 顧問裡

EV_SPECS = [
    EVSpec(model="bZ4X", display="TOYOTA bZ4X", km_per_kwh=5.8, battery_kwh=71.4, csv_keywords=("補充",), csv_path=SUPP_CSV_PATH,
           warranty="8 年原廠電池保固", lifestyle="大空間、適合假日全家出遊、V2L 露營供電"),
    # ↓ 範例：新增第二台電動車（車名與數字都是示意，請換成原廠規格後再取消註解）
    # EVSpec(model="EV_X", display="TOYOTA EV_X", km_per_kwh=6.0, battery_kwh=58.0, maint_per_year=6000,
    #        warranty="（原廠保固說法）", lifestyle="（主打的生活情境）"),
]
DEFAULT_EV = "bZ4X"                       # 儀表板 / AI 顧問預設拿來比較的電動車款
EV_CATALOG_FILE = "ev_catalog.json"       # 選用：放在 CSV 同一個資料夾就會被讀取；None = 不讀

# Gemini API Key 讀取順序：Colab 密鑰 "api" → 下面這個變數 → 環境變數 GEMINI_API_KEY / GOOGLE_API_KEY
# ⚠️ 請不要把真正的金鑰直接寫在筆記本裡再上傳到 GitHub
GEMINI_API_KEY = ""
GEMINI_MODELS = ["gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.5-flash", "gemini-3.1-flash-lite", "gemini-flash-latest"]  # 依序嘗試；成功過的模型之後會自動排到最前面（gemini-2.5-flash 對新帳號已停用，所以不放）
GEMINI_TIMEOUT_SEC = 30      # 單次請求逾時秒數；伺服器忙線(503)時不要讓介面卡太久，直接換下一個模型

In [3]:
#---------------------------------------------------------
# 【區塊 1：讀取兩份 CSV 檔案】
# 兩份檔案都是 Big5 編碼（不是 UTF-8），欄位名稱還夾有換行，所以要自動判斷編碼 + 正規化欄名
# ---------------------------------------------------------
OPTIONAL_COLUMN_KEYS = {"visits_paid_1y": "近一年有費回廠次數"}   # 選用：客群分群優先用這欄；找不到就退回「近一年回廠次數」

COLUMN_KEYS = {
    "sex": "性別", "age": "年齡", "city": "居住縣市",
    "visits_1y": "近一年回廠次數",
    "svc_amt_1y": "近一年定保消費金額",
    "repair_cnt_1y": "近一年維修次數",
    "model": "現保有車款",
    "deliver": "現保有車交車日期",
}

SEARCH_DIRS = (".", "/content", "/content/drive/MyDrive/toyota", "/content/drive/MyDrive")   # CSV / ev_catalog.json 的搜尋順序

def _find_csv(path, must_have, must_not=()):
    if path:
        return path
    hits = []
    for d in SEARCH_DIRS:
        for p in glob.glob(os.path.join(d, "*.csv")):
            name = os.path.basename(p)
            if all(k in name for k in must_have) and not any(k in name for k in must_not):
                hits.append(p)
    if not hits:
        raise FileNotFoundError(f"找不到檔名含 {must_have} 的 CSV，請先上傳到 Colab（檔案在雲端硬碟的話要先掛載 Drive），或在區塊 0 填入 *_CSV_PATH")
    return max(hits, key=os.path.getsize)

def _read_csv_auto(path):
    for enc in ("utf-8-sig", "big5", "cp950"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"無法判斷 {path} 的編碼")

def _load_cdp(path):
    raw = _read_csv_auto(path)
    raw.columns = [re.sub(r"\s+", "", str(c)) for c in raw.columns]   # 去掉欄名裡的換行/空白
    out = pd.DataFrame()
    for key, kw in COLUMN_KEYS.items():
        hits = [c for c in raw.columns if kw in c]
        if len(hits) != 1:
            raise ValueError(f"{os.path.basename(path)}：找不到唯一欄位「{kw}」，目前欄位：{list(raw.columns)}")
        out[key] = raw[hits[0]]
    for key, kw in OPTIONAL_COLUMN_KEYS.items():
        hits = [c for c in raw.columns if kw in c]
        out[key] = raw[hits[0]] if len(hits) == 1 else np.nan
    for c in ["age", "visits_1y", "visits_paid_1y", "svc_amt_1y", "repair_cnt_1y"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out["deliver"] = pd.to_datetime(out["deliver"], errors="coerce")
    out["model"] = out["model"].fillna("").astype(str).str.strip()
    out = out[~out["model"].isin(["", "nan", "NaN", "None"])].reset_index(drop=True)   # 車款空白的列無法歸類，直接排除
    out["age"] = out["age"].where(out["age"].between(18, 100))        # 0 = 未提供、>100 = 異常值，一律視為缺值
    out["svc_amt_1y"] = out["svc_amt_1y"] * A_CURRENCY_TO_NTD         # A國幣 → 台幣
    return out

def _find_csv_or_mount_drive(path, must_have, must_not=()):
    """先在 Colab 本機找；找不到就自動掛載 Google 雲端硬碟再找（CSV 放在「我的雲端硬碟/toyota/」就不用每次重傳）。"""
    try:
        return _find_csv(path, must_have, must_not)
    except FileNotFoundError:
        try:
            from google.colab import drive
            drive.mount("/content/drive")        # 新的 Colab 工作階段第一次會跳出授權視窗，點「允許」即可
        except Exception:
            pass
        return _find_csv(path, must_have, must_not)

df_main = _load_cdp(_find_csv_or_mount_drive(MAIN_CSV_PATH, ("CDP",), ("補充",)))   # 主檔：各車款車主回廠資料（檔名含「補充」的一律視為補充檔）

# ---- 電動車款目錄：區塊 0 的 EV_SPECS + 選用的 ev_catalog.json（同名覆寫、新名新增、enabled=false 下架）----
def _find_file(name):
    for d in SEARCH_DIRS:
        p = os.path.join(d, name)
        if os.path.isfile(p):
            return p
    return None

def load_ev_catalog(specs, filename=EV_CATALOG_FILE):
    catalog = {s.model: s for s in specs}
    path = _find_file(filename) if filename else None
    if path:
        allowed = {f.name for f in fields(EVSpec)}
        with open(path, encoding="utf-8") as fh:
            items = json.load(fh)
        for d in items:
            bad = set(d) - allowed
            if bad or "model" not in d:
                raise ValueError(f"{filename} 格式錯誤：每一筆都要有 model，可用欄位 {sorted(allowed)}；未知欄位 {sorted(bad)}")
            if "csv_keywords" in d:
                d["csv_keywords"] = tuple(d["csv_keywords"] or ())
            base = catalog.get(d["model"])
            try:
                catalog[d["model"]] = dc_replace(base, **d) if base else EVSpec(**d)   # 已有車款只覆寫有寫到的欄位
            except TypeError:
                raise ValueError(f"{filename}：新車款「{d['model']}」至少要填 model、display、km_per_kwh、battery_kwh") from None
        print(f"📄 已套用 {path}：{'、'.join(d['model'] for d in items)}")
    return catalog

EV_CATALOG_ALL = load_ev_catalog(EV_SPECS)                                    # 所有已知電動車款（含下架）：用來判斷「純電」，下架的也不會被誤當成燃油車
EV_CATALOG = {m: s for m, s in EV_CATALOG_ALL.items() if s.enabled}           # 上架中的電動車款：可拿來比較 / 試算
if not EV_CATALOG:
    raise ValueError("電動車款目錄是空的（或全部 enabled=False），至少要有一台上架的電動車款")
if DEFAULT_EV not in EV_CATALOG:
    print(f"⚠️ DEFAULT_EV「{DEFAULT_EV}」不在目錄裡（或已停用），改用 {next(iter(EV_CATALOG))}")
    DEFAULT_EV = next(iter(EV_CATALOG))

# ---- 各電動車款的專屬補充檔（沒有補充檔的車款直接用主檔裡的資料）----
def _load_ev_supplements():
    cache, out = {}, {}
    for m, ev in EV_CATALOG.items():
        if not ev.csv_keywords and not ev.csv_path:
            continue
        others = set().union(*[set(o.csv_keywords) for k, o in EV_CATALOG_ALL.items() if k != m]) - set(ev.csv_keywords)
        path = _find_csv_or_mount_drive(ev.csv_path, ev.csv_keywords, tuple(others))   # 排除「別台車專屬」的補充檔
        if path not in cache:
            cache[path] = _load_cdp(path)
        df = cache[path]
        rows = df[df["model"] == m].reset_index(drop=True)
        if rows.empty:
            raise ValueError(f"{os.path.basename(path)} 裡找不到車款「{m}」（檔內車款：{df['model'].value_counts().head(10).to_dict()}），"
                             "請確認 EVSpec 的 model 和 CSV 寫法一致")
        if len(rows) < len(df):
            print(f"ℹ️ {os.path.basename(path)} 有 {len(df) - len(rows):,} 筆不是 {m}，{m} 只取自己的 {len(rows):,} 筆")
        out[m] = rows
    return out

EV_SUPP = _load_ev_supplements()     # {車款: 該車款補充檔的資料}
_in_main = set(df_main["model"])
_src_note = "、".join(f"{m}（{'補充檔' if m in EV_SUPP else '主檔' if m in _in_main else '尚無車主資料'}）" for m in EV_CATALOG)
print(f"✅ CSV 資料庫讀取成功！主檔 {len(df_main):,} 筆" + "".join(f"、{m} 補充檔 {len(d):,} 筆" for m, d in EV_SUPP.items()))
print(f"🔋 電動車款目錄：{_src_note}；預設比較 {DEFAULT_EV}")

✅ CSV 資料庫讀取成功！主檔 1,048,575 筆、bZ4X 補充檔 3,037 筆
🔋 電動車款目錄：bZ4X（補充檔）；預設比較 bZ4X


In [4]:
# ---------------------------------------------------------
# 【區塊 2：從 CSV 統計各車款保養費 + 5年 TCO 養車費用預估】
# ---------------------------------------------------------
# 主檔剛好停在 Excel 列數上限(1,048,575)，部分車款不完整 → 有補充檔的車款（目前是 bZ4X）一律用補充檔，其他車款用主檔
df_all = pd.concat([df_main[~df_main["model"].isin(list(EV_SUPP))], *EV_SUPP.values()], ignore_index=True)
SNAPSHOT = df_all["deliver"].max()                                    # 資料截止日（以最晚交車日代表）
df_all["years"] = (SNAPSHOT - df_all["deliver"]).dt.days / 365.25     # 截至資料截止日的車齡

lo, hi = MAINT_AGE_BAND
EV_MODELS = list(EV_CATALOG)          # 目錄順序 = 介面下拉選單順序

def is_ev(model):
    return model in EV_CATALOG_ALL     # 含下架車款：下架只是不拿來比較，車主仍算電動車主

def get_ev(ev_model=None):
    """取得電動車款規格；None / 空白 = DEFAULT_EV。所有和電動車有關的計算都從這裡拿參數，不再讀全域常數。"""
    m = ev_model or DEFAULT_EV
    if m not in EV_CATALOG:
        raise KeyError(f"電動車款「{m}」不在目錄裡，可用：{EV_MODELS}")
    return EV_CATALOG[m]

def _powertrain(model):
    m = model.upper()
    if is_ev(model):
        return "純電"
    return "油電混合" if ("HV" in m or m.endswith("-H") or m.startswith("PRIUS")) else "汽油"

def _robust_mean(s, q=0.99):
    s = s.dropna()
    return float(s.clip(upper=s.quantile(q)).mean()) if len(s) else float("nan")   # 蓋掉最高 1% 極端值再平均

df_all["powertrain"] = df_all["model"].map(_powertrain)
_band = df_all[df_all["years"].between(lo, hi)]
POOLED_MAINT = {pt: _robust_mean(g["svc_amt_1y"]) for pt, g in _band.groupby("powertrain")}   # 樣本不足時的備援：同動力型式（汽油 / 油電混合 / 純電）平均

def _fallback_maint(model, powertrain):
    """樣本不足時的年定保費 → (金額, 來源)。電動車款優先用目錄的 maint_per_year，其次用同動力型式平均。"""
    ev = EV_CATALOG_ALL.get(model)
    if ev is not None and ev.maint_per_year is not None:
        return float(ev.maint_per_year), "spec"
    v = POOLED_MAINT.get(powertrain, float("nan"))
    if model in EV_CATALOG and pd.isna(v):
        raise ValueError(f"電動車款「{model}」車主資料不足、也沒有其他電動車款可參考，請在 EVSpec 填 maint_per_year")
    return v, "pooled"

def _build_model_stats():
    rows = []
    for model, g in df_all.groupby("model"):
        band = g[g["years"].between(lo, hi)]
        enough = len(band) >= MIN_SAMPLE
        maint, src = (_robust_mean(band["svc_amt_1y"]), "csv") if enough else _fallback_maint(model, g["powertrain"].iloc[0])
        rows.append({
            "model": model,
            "powertrain": g["powertrain"].iloc[0],
            "owners": len(g),
            "median_years": g["years"].median(),
            "n_band": len(band),
            "maint_per_year": maint,
            "maint_fallback": not enough,
            "maint_source": src,          # csv = 該車款自己的資料；pooled = 同動力型式平均；spec = 目錄假設值
            "visits_band": _robust_mean(band["visits_1y"]) if enough else float("nan"),
            "repair_band": _robust_mean(band["repair_cnt_1y"]) if enough else float("nan"),
        })
    have = set(df_all["model"])
    for model in EV_MODELS:              # 目錄裡有、CSV 還沒有車主的新車款：只補保養費（假設值），其餘統計留空
        if model not in have:
            maint, src = _fallback_maint(model, "純電")
            rows.append({"model": model, "powertrain": "純電", "owners": 0, "median_years": float("nan"), "n_band": 0,
                         "maint_per_year": maint, "maint_fallback": True, "maint_source": src,
                         "visits_band": float("nan"), "repair_band": float("nan")})
    return pd.DataFrame(rows).set_index("model")

MODEL_STATS = _build_model_stats()
EV_STATS  = MODEL_STATS.loc[EV_MODELS]                            # 所有電動車款（DataFrame，一列一台）
ICE_STATS = MODEL_STATS[~MODEL_STATS.index.isin(list(EV_CATALOG_ALL))]   # 燃油 / 油電車款（排除所有電動車款，含下架）

def maint_src_text(model):
    """定保費的資料來源說明（儀表板與 AI 顧問共用，口徑一致）。"""
    r = MODEL_STATS.loc[model]
    if r["maint_source"] == "csv":
        return f"{model} 車齡{lo}-{hi}年車主 {int(r['n_band']):,} 位"
    if r["maint_source"] == "pooled":
        return f"{model} 車齡{lo}-{hi}年樣本不足，採同為{r['powertrain']}車款的平均"
    return f"{model} 尚無足夠車主資料，採目錄設定的假設值"

print("🔋 電動車款定保費：" + "；".join(
    f"{m} NT${r['maint_per_year']:,.0f}/年（{ {'csv': 'CSV', 'pooled': '同動力平均', 'spec': '假設值'}[r['maint_source']] }，車主 {int(r['owners']):,} 位）"
    for m, r in EV_STATS.iterrows()))
CAR_CHOICES = ICE_STATS[ICE_STATS["owners"] >= MIN_SAMPLE].sort_values("owners", ascending=False).index.tolist()

def calculate_tco(current_car_type, annual_km, years=YEARS, elec_price=None, ev_model=None):
    s = ICE_STATS.loc[current_car_type]
    ev = get_ev(ev_model)
    e = EV_STATS.loc[ev.model]
    km_per_l = FUEL_KM_PER_L_BY_MODEL.get(current_car_type, FUEL_KM_PER_L[s["powertrain"]])

    ice_energy_cost = (annual_km / km_per_l) * GAS_PRICE_PER_L * years
    ev_energy_cost  = (annual_km / ev.km_per_kwh) * (ELEC_PRICE_PER_KWH if elec_price is None else elec_price) * years   # elec_price 由充電情境算出的平均電價帶入
    ice_maint_cost  = s["maint_per_year"] * years          # ← 來自 CSV
    ev_maint_cost   = e["maint_per_year"] * years          # ← 來自 CSV（該電動車款；樣本不足時見 maint_source）

    total_ice = ice_energy_cost + ice_maint_cost
    total_ev  = ev_energy_cost + ev_maint_cost
    saved_money = total_ice - total_ev
    detail = {
        "km_per_l": km_per_l, "powertrain": s["powertrain"],
        "ice_maint_yr": s["maint_per_year"], "ice_n": int(s["n_band"]), "ice_fallback": bool(s["maint_fallback"]),
        "ev_model": ev.model, "ev_display": ev.display, "ev_km_per_kwh": ev.km_per_kwh,
        "ev_maint_yr": e["maint_per_year"], "ev_n": int(e["n_band"]), "ev_fallback": bool(e["maint_fallback"]),
        "ice_energy": ice_energy_cost, "ev_energy": ev_energy_cost,      # 5 年能源費（給 AI 顧問引用）
        "ice_maint": ice_maint_cost, "ev_maint": ev_maint_cost,          # 5 年保養費（給 AI 顧問引用）
    }
    return total_ice, total_ev, saved_money, detail

# ---------------------------------------------------------
# 【區塊 2.2：充電需求推估】（規則式估算，不是訓練出來的模型；所有參數都在區塊 0）
# 同樣的年里程，有沒有固定車位會讓平均電價差很多，所以 TCO 的電費也改用這裡算出的平均電價
# ---------------------------------------------------------
def charging_plan(annual_km, has_home_charger, ev_model=None):
    ev = get_ev(ev_model)
    annual_kwh = annual_km / ev.km_per_kwh
    home_share = HOME_SHARE[bool(has_home_charger)]
    price = home_share * ELEC_PRICE_PER_KWH + (1 - home_share) * PUBLIC_CHARGE_PRICE_PER_KWH
    per_charge_km = ev.battery_kwh * (CHARGE_WINDOW[1] - CHARGE_WINDOW[0]) * ev.km_per_kwh
    charges_per_week = (annual_km / 52) / per_charge_km
    return {"ev_model": ev.model, "battery_kwh": ev.battery_kwh, "annual_kwh": annual_kwh, "monthly_kwh": annual_kwh / 12, "price": price, "home_share": home_share,
            "monthly_cost": annual_kwh / 12 * price, "per_charge_km": per_charge_km,
            "charges_per_week": charges_per_week, "days_between": 7 / max(charges_per_week, 1e-9)}

def charge_text(plan, home):
    return (f"- 充電需求推估（規則式估算）：每月約 {plan['monthly_kwh']:,.0f} 度、電費約 NT${plan['monthly_cost']:,.0f}/月（平均電價 NT${plan['price']:.1f}/度，"
            f"{'有固定車位，約 %d%% 在家充' % round(plan['home_share'] * 100) if home else '無固定車位，主要靠公共充電'}）；"
            f"電量 {CHARGE_WINDOW[0]:.0%}→{CHARGE_WINDOW[1]:.0%} 算一次充電（約可行駛 {plan['per_charge_km']:,.0f} 公里），"
            f"平均每週約充 {plan['charges_per_week']:.1f} 次（約每 {plan['days_between']:.0f} 天一次）。"
            f"{plan['ev_model']} 電池容量 {plan['battery_kwh']:g} kWh、公共充電價 NT${PUBLIC_CHARGE_PRICE_PER_KWH:g}/度皆為假設值。")

def trip_check(trip_km, ev_model=None):
    """長途情境（規則式）：單趟一次到不到？不到的話中途要補電幾次？來回（目的地沒有充電）又要幾次？"""
    ev = get_ev(ev_model)
    km_per_kwh = ev.km_per_kwh * HIGHWAY_RANGE_FACTOR
    leg1 = ev.battery_kwh * (TRIP_START_SOC - TRIP_ARRIVE_SOC) * km_per_kwh        # 出發電量 → 保留電量 可跑的距離
    leg_next = ev.battery_kwh * (TRIP_CHARGE_TO - TRIP_ARRIVE_SOC) * km_per_kwh    # 每次中途補電後可再跑的距離
    stops = lambda d: 0 if d <= leg1 else int(np.ceil((d - leg1) / leg_next))
    return {"leg1": leg1, "leg_next": leg_next, "oneway_stops": stops(trip_km), "round_stops": stops(2 * trip_km)}

def trip_text(trip_km, ev_model=None):
    if not trip_km or trip_km <= 0:
        return "- 長途情境：車主未提供常跑的最長單趟距離"
    ev = get_ev(ev_model)
    t = trip_check(float(trip_km), ev.model)
    one = "單趟不需中途充電" if t["oneway_stops"] == 0 else f"單趟需中途補電約 {t['oneway_stops']} 次"
    rnd = "來回也不需中途充電" if t["round_stops"] == 0 else f"若目的地沒有充電設施，來回需補電約 {t['round_stops']} 次"
    return (f"- 長途情境檢查（規則式估算，以 {ev.model} 計）：常跑的最長單趟約 {trip_km:g} 公里。出發電量 {TRIP_START_SOC:.0%}、抵達保留 {TRIP_ARRIVE_SOC:.0%}、"
            f"高速行駛續航打 {HIGHWAY_RANGE_FACTOR:.0%} 折時，一趟約可跑 {t['leg1']:,.0f} 公里 → {one}；{rnd}"
            f"（每次補電到 {TRIP_CHARGE_TO:.0%} 約可再跑 {t['leg_next']:,.0f} 公里）。續航折扣與電量門檻皆為假設值，實際以原廠數據與路況為準。")

🔋 電動車款定保費：bZ4X NT$696/年（CSV，車主 3,037 位）


In [5]:
# ---------------------------------------------------------
# 【區塊 2.5：客群分群（KMeans）→ 給 AI 顧問的「內部話術方向」】
# 用 CDP 回廠資料把「燃油/油電車主」分成 3 群，再依客人的車齡、回廠次數、年里程挑話術方向。
# 和直接貼上來的版本相比，刻意做了三處不同（原因見註解）：
#   1. 群的名字由「各群中心的數值」決定，不寫死 cluster 編號（KMeans 的編號是任意的，換資料/亂數種子就會變）
#   2. 預測時只拿客人真的提供的欄位（車齡、回廠次數）去比各群中心，不用全體平均補另外兩個欄位（會讓判斷偏掉）
#   3. 沿用上面已載入的 df_all，不再重新讀一次 100 萬列的 CSV
# ---------------------------------------------------------
if SEG_K != 3:
    raise ValueError("目前的話術只定義了 3 種客群（高頻 / 低頻老車 / 標準），SEG_K 請維持 3")

_seg_src = df_all[~df_all["model"].isin(list(EV_CATALOG_ALL))]   # 目標是「油轉電」客群，所以只用燃油 / 油電車主訓練
_paid_ratio = _seg_src["visits_paid_1y"].notna().mean() if "visits_paid_1y" in _seg_src else 0.0
SEG_VISITS_COL = "visits_paid_1y" if _paid_ratio >= 0.5 else "visits_1y"   # 優先用「有費回廠次數」，沒有才退回「回廠次數」
_seg = pd.DataFrame({"car_age": _seg_src["years"], "visits": _seg_src[SEG_VISITS_COL],
                     "svc_amt": _seg_src["svc_amt_1y"], "repairs": _seg_src["repair_cnt_1y"]}).dropna()
_seg = _seg[(_seg["visits"] <= 24) & (_seg["repairs"] <= 24) & (_seg["car_age"] >= 0)]
_seg["svc_amt"] = _seg["svc_amt"].clip(upper=_seg["svc_amt"].quantile(0.99))   # 蓋掉最高 1% 的極端花費
if len(_seg) < SEG_K * 100:
    raise ValueError(f"可用於分群的資料只有 {len(_seg)} 筆，不足以分成 {SEG_K} 群，請檢查 CSV 欄位")
if len(_seg) > SEG_MAX_ROWS:
    _seg = _seg.sample(SEG_MAX_ROWS, random_state=42)

SEG_FEATURES = ["car_age", "visits", "svc_amt", "repairs"]
seg_scaler = StandardScaler().fit(_seg[SEG_FEATURES].to_numpy())
_X = seg_scaler.transform(_seg[SEG_FEATURES].to_numpy())
seg_km = KMeans(n_clusters=SEG_K, random_state=42, n_init=10).fit(_X)
SEG_SILHOUETTE = silhouette_score(_X, seg_km.labels_, sample_size=min(20000, len(_X)), random_state=42)

SEG_PROFILE = pd.DataFrame(seg_scaler.inverse_transform(seg_km.cluster_centers_), columns=SEG_FEATURES)   # 各群中心（原始單位）
SEG_PROFILE["owners"] = np.bincount(seg_km.labels_, minlength=SEG_K)
SEG_PROFILE["share"] = SEG_PROFILE["owners"] / SEG_PROFILE["owners"].sum()

def _roles_from_profile(profile):
    """依各群中心決定「這一群是哪種客群」：回廠最頻繁的 = 高頻；其餘兩群中較老的 = 低頻老車；剩下的 = 標準。"""
    order = profile["visits"].sort_values(ascending=False).index.tolist()
    old = max(order[1:], key=lambda c: profile.loc[c, "car_age"])
    return {order[0]: "high", old: "old_low", [c for c in order[1:] if c != old][0]: "standard"}

SEG_ROLE = _roles_from_profile(SEG_PROFILE)

SEG_LABEL = {
    "high": "高頻商務/長途族", "old_low": "低頻用車老車族", "standard": "標準家庭通勤族",
    "outside": "里程極高但極少回原廠（疑似外廠保養）",
}
# ⚠️ 下面話術裡的產品說法（電池保固、生活情境賣點、主被動安全、電池檢修）不在 CSV 資料庫裡，
#    請先向原廠 / 經銷商確認，再決定要不要對外使用。
#    {ev} {warranty} {lifestyle} 會依「比較的電動車款」從 EVSpec 帶入，換車款時話術自動跟著換；共通的說法改這裡。
SEG_STRATEGY = {
    "outside": "這位客人『里程極高，但極少回原廠』，可能在外廠保養。可主打『{ev} {warranty}』，"
               "並說明電池與高壓系統的檢修建議由原廠技師處理，藉此把客人拉回原廠體系。",
    "old_low": "這位客人是『低頻用車老車族』。請主打換購後的『主被動安全升級』與『不用擔心老車隨時拋錨的安心感』。",
    "high": "這位客人是『高頻商務/長途族』。請強烈強調保養省下的金額，並主打『省下頻繁保養的寶貴時間』。",
    "standard": "這位客人是『標準家庭通勤族』。請主打『{lifestyle}』的生活情境。",
}

def strategy_text(role, ev_model=None):
    ev = get_ev(ev_model)
    return SEG_STRATEGY[role].format(ev=ev.model,
                                     warranty=ev.warranty or "原廠電池保固（年限請向原廠確認）",
                                     lifestyle=ev.lifestyle or "日常通勤省油錢、安靜舒適")

_KNOWN = [SEG_FEATURES.index("car_age"), SEG_FEATURES.index("visits")]

def predict_cluster(car_age, visits):
    """只在客人提供的欄位（車齡、回廠次數）上比對各群中心。
    不要用全體平均去補『花費、維修』再丟進 kmeans.predict：高頻群的花費、維修都偏高，補成平均值會讓它被扣分，
    結果明明一年回廠很多次，也很少被判成高頻群。"""
    z = (np.array([car_age, visits], dtype=float) - seg_scaler.mean_[_KNOWN]) / seg_scaler.scale_[_KNOWN]
    return int(((seg_km.cluster_centers_[:, _KNOWN] - z) ** 2).sum(axis=1).argmin())

def segment_customer(car_age, visits, annual_km, ev_model=None):
    cid = predict_cluster(car_age, visits)
    role, rule = SEG_ROLE[cid], ""
    if annual_km > OUTSIDE_SHOP_KM and visits <= OUTSIDE_SHOP_VISITS:     # 規則優先：里程高卻幾乎不回廠
        role = "outside"
        rule = f"觸發規則：年里程 > {OUTSIDE_SHOP_KM:,} 公里，且近一年回廠 ≤ {OUTSIDE_SHOP_VISITS} 次"
    return {"cluster": cid, "model_role": SEG_ROLE[cid], "role": role, "label": SEG_LABEL[role],
            "strategy": strategy_text(role, ev_model), "rule": rule}

def seg_context(car, car_age, visits, annual_km, ev_model=None):
    """寫進 Gemini prompt 的內部話術方向（已經是電動車主不需要換購話術）。"""
    if is_ev(car):
        return ""
    s = segment_customer(car_age, visits, annual_km, ev_model)
    return ("【內部客群診斷（只用來調整說話重點；不要向客戶提到「客群」「分群」「診斷」「戰略」，也不要逐字照念）】\n"
            f"- 客群：{s['label']}\n- 話術方向：{s['strategy']}\n\n")

def segment_report(car, car_age, visits, annual_km, ev_model=None):
    """業務端驗證用：顯示這位客人被分到哪一群、為什麼，以及各群的樣貌。不會出現在給客戶看的對話裡。"""
    if is_ev(car):
        return f"您已經是 **{car}** 車主，不需要做換購客群診斷。"
    car_age, visits, annual_km = float(car_age or 0), float(visits or 0), int(annual_km or 0)
    s = segment_customer(car_age, visits, annual_km, ev_model)
    rows = ["| 客群 | 人數占比 | 平均車齡(年) | 平均近一年回廠(次) | 平均年定保花費 | 平均近一年維修(次) |", "|---|---|---|---|---|---|"]
    for cid, r in SEG_PROFILE.iterrows():
        mark = "👉 " if cid == s["cluster"] else ""
        rows.append(f"| {mark}{SEG_LABEL[SEG_ROLE[cid]]} | {r['share']:.0%} | {r['car_age']:.1f} | {r['visits']:.2f} | {r['svc_amt']:,.0f} | {r['repairs']:.2f} |")
    return (f"**判斷結果**：{s['label']}\n\n"
            f"- 依車齡 {car_age:g} 年、近一年回廠 {visits:g} 次，最接近「{SEG_LABEL[s['model_role']]}」\n"
            + (f"- {s['rule']}\n" if s["rule"] else "")
            + f"- 話術方向：{s['strategy']}\n\n"
            + "\n".join(rows)
            + f"\n\n<sub>分群訓練 {len(_seg):,} 筆（回廠次數欄位：{SEG_VISITS_COL}）、silhouette = {SEG_SILHOUETTE:.2f}"
              "（越接近 1 分得越清楚；群與群之間本來就有重疊，請當作「說話重點的參考」而不是定論）。"
              "客群名稱是依回廠資料做的推測。</sub>")

_t = SEG_PROFILE.copy()
_t.insert(0, "客群", [SEG_LABEL[SEG_ROLE[c]] for c in _t.index])
_t["share"] = (_t["share"] * 100).round(1).astype(str) + "%"
print(f"✅ 客群分群完成：訓練 {len(_seg):,} 筆、回廠次數欄位 = {SEG_VISITS_COL}、silhouette = {SEG_SILHOUETTE:.2f}")
print(_t.round({"car_age": 1, "visits": 2, "svc_amt": 0, "repairs": 2}).to_string())
print("👆 請看一下各群的數字是否合理（例如『高頻』那群回廠次數最多、『低頻老車』車齡最大），不合理就調整 SEG_* 參數或話術。")

✅ 客群分群完成：訓練 300,000 筆、回廠次數欄位 = visits_paid_1y、silhouette = 0.40
         客群  car_age  visits  svc_amt  repairs  owners  share
0   低頻用車老車族     11.9    0.17     68.0     0.07  160030  53.3%
1  高頻商務/長途族      5.1    3.89   2043.0     2.14   31127  10.4%
2   標準家庭通勤族      4.7    1.72   1181.0     0.37  108843  36.3%
👆 請看一下各群的數字是否合理（例如『高頻』那群回廠次數最多、『低頻老車』車齡最大），不合理就調整 SEG_* 參數或話術。


In [6]:
# ---------------------------------------------------------
# 【區塊 3：Gemini AI 顧問】
# 100 萬列 CSV 不能整份塞進 prompt，改餵「CSV 統計摘要」+「該車主的檔案與精算結果」
# AI 回覆為 JSON：文字回答 + 顧慮分類，全部用文字回答（不產生圖表）
# ---------------------------------------------------------
def _load_api_key():
    try:
        from google.colab import userdata
        key = userdata.get("api")
        if key:
            return key
    except Exception:
        pass
    return GEMINI_API_KEY or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY") or ""

API_KEY = _load_api_key()
client = genai.Client(api_key=API_KEY, http_options=types.HttpOptions(timeout=GEMINI_TIMEOUT_SEC * 1000)) if API_KEY else None   # 沒有金鑰也能先開介面，對話時會提示

def _fmt(x, nd=1):
    return "樣本不足" if pd.isna(x) else f"{x:.{nd}f}"   # 不加千分位，避免和 CSV 的逗號分隔混淆

def _ev_profile_text(m):
    """單一電動車款的車主輪廓 + 試算用規格（假設值）。目錄裡每台 EV 各一段，新增車款不用改這裡。"""
    spec, ev = EV_CATALOG[m], df_all[df_all["model"] == m]
    head = f"{m}（{spec.display}）試算用規格（假設值）：電耗約 {spec.km_per_kwh:g} km/kWh、電池 {spec.battery_kwh:g} kWh\n"
    if ev.empty:
        return head + f"- 資料庫尚無 {m} 車主資料（新車款），定保費用為{'目錄假設值' if MODEL_STATS.loc[m, 'maint_source'] == 'spec' else '其他電動車款平均'}"
    sex = ev["sex"].where(ev["sex"].isin(["男", "女"]), "其他/未提供").value_counts(normalize=True)
    top_city = ev.loc[ev["city"] != "99_other", "city"].value_counts().head(5)
    by_year = ev["deliver"].dt.year.value_counts().sort_index()
    return head + (
        f"{m} 車主{'補充' if m in EV_SUPP else '主檔'}資料（共 {len(ev):,} 位，資料截止日 {SNAPSHOT:%Y-%m-%d}）：\n"
        f"- 性別：{'、'.join(f'{k}{v:.0%}' for k, v in sex.items())}；年齡中位數 {ev['age'].median():.0f} 歲\n"
        f"- 居住縣市前五名：{'、'.join(f'{k}{v:,}人' for k, v in top_city.items())}\n"
        f"- 交車年度：{'、'.join(f'{int(k)}年{v:,}人' for k, v in by_year.items())}\n"
        f"- 近一年平均回廠 {_robust_mean(ev['visits_1y']):.1f} 次、平均維修 {_robust_mean(ev['repair_cnt_1y']):.1f} 次"
    )

def _maint_cell(r):
    if r["maint_source"] == "csv":
        return _fmt(r["maint_per_year"], 0)
    return f"假設值{_fmt(r['maint_per_year'], 0)}" if r["maint_source"] == "spec" else "樣本不足"

def build_knowledge_text():
    db1 = "電動車款資料：\n" + "\n\n".join(_ev_profile_text(m) for m in EV_MODELS)
    lines = [f"車款,動力,車主數,車齡中位數(年),車齡{lo}-{hi}年樣本數,車齡{lo}-{hi}年_年均定保花費(NT$),車齡{lo}-{hi}年_近一年平均回廠次數,車齡{lo}-{hi}年_近一年平均維修次數"]
    for m, r in MODEL_STATS.sort_values("owners", ascending=False).iterrows():
        lines.append(f"{m},{r['powertrain']},{r['owners']},{r['median_years']:.1f},{r['n_band']},"
                     f"{_maint_cell(r)},{_fmt(r['visits_band'])},{_fmt(r['repair_band'])}")
    db2 = "各車款車主回廠統計（主檔 {:,} 筆彙整）：\n".format(len(df_main)) + "\n".join(lines)
    return db1, db2

DB1_TEXT, DB2_TEXT = build_knowledge_text()

SYSTEM_RULES = """
你是 TOYOTA 專屬的「SmartLife 智能隨行顧問 × EV LifePilot 油轉電 AI 顧問」。
你的任務是消除車主對電動車 (__EV_NAMES__) 的疑慮，並協助規劃用車成本。
請使用繁體中文，語氣親切、專業、有同理心，善用白話比喻。

【資料使用規則】
1. 請【嚴格根據】下方三份資料庫，以及每次提問附上的「車主檔案」與「系統精算結果」回答；引用 CSV 數字時請說明是「車主回廠資料統計」；引用 DB3 市調數字時請說明是「Kantar 2025 W3 品牌調查（一般大眾，非 TOYOTA 車主）」，
   且只能說「有 X% 的受訪者擔心…」，不可說成這位車主或所有車主的想法。
2. 資料庫沒有的內容（例如電池保固年限、充電樁安裝規定與補助、車價、原廠優惠）請誠實告知資料庫未提供，
   只給一般性的建議，並請客戶向 TOYOTA 經銷商確認，不可編造數字、保固條款或確切價格。
3. 費用試算的油價、電價、油耗、電耗為「假設值」，回答時須說明假設，並提醒實際以原廠報價為準。
4. 若提問附有【內部客群診斷】，請依其「話術方向」調整說話重點與語氣，但不要向客戶提到「客群」「分群」「診斷」「戰略」等字眼，也不要逐字照念。
   話術方向裡由公司提供的產品說法（如保固年限、功能、維修規定）可以引用，但必須註明「實際以 TOYOTA 原廠公告 / 經銷商說明為準」。

【輸出格式】請「只」輸出一個 JSON 物件，不要有其他文字：
{
  "answer": "給車主的文字回答（可用 Markdown 條列）",
  "concern": "充電" | "電池" | "價格" | "里程" | "維修保養" | "技術疑慮" | "其他" | null
}
回答規則：
1. 一律用文字回答，不要產生圖表。數字比較、費用試算、保養時程、長期成本、用電/油耗比較等，請用文字說明或 Markdown 條列把數字講清楚；
   即使車主要求「畫圖」「用圖表比較」，也改用文字條列呈現，不要說「如下圖」。
2. 引用的數字必須來自資料庫或「系統精算結果」；需要推算時，請在 answer 中說明假設（如年里程、油電價）。

顧慮判斷規則：
1. concern 請判斷車主「這一則提問」背後最主要的換購顧慮：
   充電（充電樁、充電時間、充電速度）、電池（更換費用、壽命、保固）、價格（購車價、成本、中古車價）、
   里程（續航不夠用、長途）、維修保養（保養不方便、維修成本）、技術疑慮（技術不成熟、安全）、其他；單純打招呼或看不出顧慮時填 null。
   這七類對應 DB3 市調中一般大眾「不考慮購買 BEV 的原因」。
2. 判斷出顧慮後，answer 請先用一句話同理並點出你理解的擔憂，再用「車主檔案」與「系統精算結果」裡的數字（例如充電需求推估、長途情境檢查）具體回應，不要只講通則。
"""

DB3_TEXT = """Kantar 2025 W3 TOYOTA 品牌調查（一般大眾，非 TOYOTA 車主；僅供 2026 和泰 AI 黑客松使用）：
- 動力形式認知度（N=1000）：油電 HEV 92%、純電 BEV 87%、PHEV 60%、FCEV 38%
- 考慮購買的 BEV 品牌（知道 BEV 者，N=867，可複選）：T**LA 37%、TOYOTA 34%、H**DA 15%；不考慮購買 BEV 者 26%（較上波 +5 個百分點）
- 不考慮購買 BEV 的原因（N=225，可複選）：擔心充電不方便 66%、更換電池費用昂貴 58%、擔心電池壽命短 56%、車款價格較高 41%、
  擔心技術尚不成熟 40%、維修保養不方便 40%、擔心行駛里程不敷使用 40%（較上波 +11 個百分點）、充電速度較慢 34%、
  擔心保固不佳 20%、馬力不足 14%、擔心中古車價 9%、不安全 1%"""

SYSTEM_INSTRUCTION = (
    SYSTEM_RULES.replace("__EV_NAMES__", "、".join(EV_MODELS))     # 車款名稱來自目錄，不寫死
    + f"\n金額原始單位為 A 國幣，本系統以 1 A國幣 = {A_CURRENCY_TO_NTD:g} 元新台幣換算後呈現。\n"
    + f"\n【資料庫 1】\n{DB1_TEXT}\n\n【資料庫 2】\n{DB2_TEXT}\n\n【資料庫 3】\n{DB3_TEXT}\n"
)

_MODEL_ORDER = list(GEMINI_MODELS)

def _ask_gemini(system_instruction, prompt):
    if client is None:
        raise ValueError("尚未設定 Gemini API Key：請在 Colab 左側「密鑰」新增名為 api 的 secret，或在區塊 0 填入 GEMINI_API_KEY")
    import time
    last_err = None
    for attempt in range(2):                                 # 全部模型都忙線(503)時，隔 2 秒再整輪試一次
        for name in list(_MODEL_ORDER):
            try:
                resp = client.models.generate_content(
                    model=name, contents=prompt,
                    config=types.GenerateContentConfig(
                        system_instruction=system_instruction,
                        response_mime_type="application/json"))
                if name != _MODEL_ORDER[0]:                  # 記住可用的模型，下次直接用
                    _MODEL_ORDER.remove(name)
                    _MODEL_ORDER.insert(0, name)
                return resp.text or ""
            except Exception as e:
                last_err = e
        time.sleep(2)
    raise last_err

# 取出 Chatbot 歷史訊息中的純文字（不同 Gradio 版本 content 可能是 str / list / dict / 元件）
def _text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(c.get("text", "") if isinstance(c, dict) else (c if isinstance(c, str) else "") for c in content)
    if isinstance(content, dict):
        return content.get("text", "")
    return ""   # 其他元件不放進文字紀錄

CONCERN_TYPES = ("充電", "電池", "價格", "里程", "維修保養", "技術疑慮", "其他")   # 對應 DB3 市調「不考慮購買 BEV 的原因」

def parse_reply(raw):
    raw = (raw or "").strip()
    candidates = [raw]
    m = re.search(r"\{.*\}", raw, re.S)                       # 模型偶爾會包 ```json 或多講一句話
    if m:
        candidates.append(m.group(0))
    for c in candidates:
        try:
            data = json.loads(c)
        except Exception:
            continue
        if isinstance(data, dict):
            ans = data.get("answer")
            ans = "" if ans is None else ("\n".join(map(str, ans)) if isinstance(ans, list) else str(ans))   # null / 清單都轉成可讀文字
            concern = data.get("concern")
            return ans, (concern if concern in CONCERN_TYPES else None)
    return raw, None

def tco_context(car, annual_km, home=True, ev_model=None):
    """把目前車款 + 年里程的精算結果寫成文字，讓 AI 顧問的數字和儀表板一致。"""
    if is_ev(car):
        return f"車主目前已經駕駛 {car}（車主資料 {int(MODEL_STATS.loc[car, 'owners']):,} 位），不需要換購試算。"
    if car not in ICE_STATS.index:
        return "（車款不在資料庫內，無法精算）"
    evm = get_ev(ev_model).model
    plan = charging_plan(annual_km, home, evm)
    ice, ev, saved, d = calculate_tco(car, annual_km, elec_price=plan["price"], ev_model=evm)
    return (
        f"- 比較的電動車款：{d['ev_display']}\n"
        f"- 假設：年行駛 {annual_km:,} 公里、{YEARS} 年；汽油 NT${GAS_PRICE_PER_L:g}/L、{car} 約 {d['km_per_l']:g} km/L；"
        f"家用電價 NT${ELEC_PRICE_PER_KWH:g}/度、{evm} 約 {d['ev_km_per_kwh']:g} km/kWh；平均電價 NT${plan['price']:.1f}/度\n"
        f"- 現有 {car}（{d['powertrain']}）{YEARS} 年：能源 NT${d['ice_energy']:,.0f} + 定保 NT${d['ice_maint']:,.0f}"
        f"（每年 NT${d['ice_maint_yr']:,.0f}）= 總計 NT${ice:,.0f}\n"
        f"- 換購 {evm} {YEARS} 年：能源 NT${d['ev_energy']:,.0f} + 定保 NT${d['ev_maint']:,.0f}"
        f"（每年 NT${d['ev_maint_yr']:,.0f}）= 總計 NT${ev:,.0f}\n"
        f"- 結論：換購 {evm} {YEARS} 年{'可省下' if saved >= 0 else '約多花'} NT${abs(saved):,.0f}\n"
        f"- 定保資料依據：{maint_src_text(car)}；{maint_src_text(evm)}\n"
        + charge_text(plan, home)
    )

def build_prompt(user_message, history, name, car, mileage, annual_km, car_age, visits, pref, home=True, trip_km=150, ev_model=None):
    lines = []
    for m in history[-12:]:                                  # 只帶最近 12 則，避免 prompt 越來越長
        if not isinstance(m, dict) or (m.get("metadata") or {}).get("status") == "pending":   # 略過「顧問思考中」暫時訊息
            continue
        t = _text(m.get("content"))
        if t and not t.startswith("💡 顧問小提醒"):
            lines.append(f"{'車主' if m.get('role') == 'user' else '顧問'}：{t}")
    return (
        f"【車主檔案】\n"
        f"- 稱呼：{name or '（未填）'}\n- 目前車款：{car}\n- 目前里程數：{mileage or '（未填）'}\n"
        f"- 預估年行駛里程：{annual_km:,} 公里\n- 目前車齡：{car_age:g} 年\n- 近一年回廠次數：{visits:g} 次\n"
        f"- 用車習慣/喜好：{pref or '（未填）'}\n- 是否有固定車位可裝充電樁：{'有' if home else '沒有'}\n- 常跑的最長單趟距離：{trip_km:g} 公里\n\n"
        f"【系統精算結果】\n{tco_context(car, annual_km, home, ev_model)}\n{trip_text(trip_km, ev_model)}\n\n"
        f"{seg_context(car, car_age, visits, annual_km, ev_model)}"
        f"--- 歷史對話紀錄 ---\n" + "\n".join(lines) + f"\n\n車主：{user_message}\n請輸出 JSON："
    )

CHAT_PLACEHOLDER   = "請輸入您的問題並按 Enter 送出..."
LOCKED_PLACEHOLDER = "請先在上方點擊「確認並建立檔案」..."
BUSY_PLACEHOLDER   = "顧問思考中，請稍候..."

def _thinking_message():
    # metadata.status="pending" 會讓 Gradio 對話框顯示轉圈圈的「思考中」氣泡，Gemini 回覆後整則被答案取代
    return {"role": "assistant", "content": "正在參考您的車主檔案與車主回廠資料統計，整理回覆中…",
            "metadata": {"title": "🤔 顧問思考中", "status": "pending"}}

def sync_chat_box(ready, busy):
    """輸入框 / 送出鈕的狀態只由兩個狀態推導：檔案是否已建立(ready)、顧問是否正在回覆(busy)。
    各處不再各改各的，才不會出現「回覆中又按確認 → 輸入框被解鎖 → 對話錯亂」這類互相打架的情形。"""
    if busy:
        return gr.update(interactive=False, placeholder=BUSY_PLACEHOLDER), gr.update(interactive=False)
    if ready:
        return gr.update(interactive=True, placeholder=CHAT_PLACEHOLDER), gr.update(interactive=True)
    return gr.update(interactive=False, placeholder=LOCKED_PLACEHOLDER), gr.update(interactive=False)

def chat_with_ai(user_message, history, name, car, mileage, annual_km, car_age, visits, pref, home, trip_km, ready, ev_model=None):
    # outputs = [輸入框, 對話框, 送出鈕, busy 狀態]；回覆結束後由 sync_chat_box 負責解鎖
    history = list(history or [])
    if not ready:                                            # 正常操作碰不到（輸入框是鎖住的），純粹保險
        gr.Warning("請先建立車主檔案")
        yield gr.update(), history, gr.update(), gr.skip()
        return
    if not user_message or not user_message.strip():
        yield gr.update(), history, gr.update(), gr.skip()
        return

    prompt = build_prompt(user_message, history, name, car, mileage, int(annual_km or 0), float(car_age or 0), float(visits or 0), pref, bool(home), float(trip_km or 0), ev_model=ev_model)
    history.append({"role": "user", "content": user_message})
    # 先顯示使用者訊息 +「顧問思考中」；等待期間鎖住輸入框與送出鈕，避免連續送出讓對話錯亂
    yield (gr.update(value="", interactive=False, placeholder=BUSY_PLACEHOLDER),
           history + [_thinking_message()], gr.update(interactive=False), True)

    try:
        answer, concern = parse_reply(_ask_gemini(SYSTEM_INSTRUCTION, prompt))
        if answer and concern:
            answer += f"\n\n<sub>🔎 系統判讀您這次最在意的是「{concern}」，已依此調整回覆重點。</sub>"
        history.append({"role": "assistant", "content": answer or "（顧問暫時沒有產生回覆，請換個問法再試一次）"})
    except Exception as e:
        history.append({"role": "assistant", "content": f"💡 顧問小提醒：發生異常，請確認 API Key 或資料庫狀態。({e})"})
    yield gr.update(), history, gr.update(), False

In [7]:
# ---------------------------------------------------------
# 【區塊 4：車主檔案 + TCO 儀表板更新函數】
# ---------------------------------------------------------
def confirm_profile(name, car, mileage, annual_km, pref):
    # outputs = [系統狀態, ready 狀態]；輸入框要不要解鎖交給 sync_chat_box 決定（回覆中不會被誤解鎖）
    if not name or not name.strip() or not car:
        return "⚠️ 請至少填寫您的稱呼與車款！", False
    if is_ev(car):
        info = f"{car}（純電，車主 {int(MODEL_STATS.loc[car, 'owners']):,} 位）"
    else:
        r = ICE_STATS.loc[car]
        info = f"{car}（{r['powertrain']}，車主 {int(r['owners']):,} 位）"
    return (f"✅ 檔案建立成功！您好，{name.strip()}。\n系統已紀錄您的愛車為 {info}、年行駛 {int(annual_km):,} 公里，請在下方開始對話。", True)

def update_dashboard(car_type, annual_km, home=True, trip_km=150, ev_model=None):
    annual_km = int(annual_km)
    if is_ev(car_type):
        return None, f"🎉 您目前駕駛的就是 **{car_type}**，不需要換購試算！想了解用車成本或保養，直接問下方的 AI 顧問吧。"
    if car_type not in ICE_STATS.index:
        return None, "⚠️ 請先選擇目前駕駛的車款。"

    ev = get_ev(ev_model)
    plan = charging_plan(annual_km, home, ev.model)
    ice_cost, ev_cost, saved, d = calculate_tco(car_type, annual_km, elec_price=plan["price"], ev_model=ev.model)

    fig = go.Figure(data=[
        go.Bar(name='現有燃油車', x=[f'{YEARS}年總花費(能源+保養)'], y=[ice_cost], marker_color='#E53935'),
        go.Bar(name=ev.display, x=[f'{YEARS}年總花費(能源+保養)'], y=[ev_cost], marker_color='#4CAF50')
    ])

    fig.update_layout(
        title=f"<b>換購 {ev.model} {YEARS} 年預估{'可為您省下' if saved >= 0 else '約多花'} NT$ {abs(saved):,.0f} 元</b>",
        barmode='group',
        height=380,
        margin=dict(l=20, r=20, t=50, b=20)
    )

    summary_text = (
        f"💡 **專屬診斷報告**：依您駕駛 **{car_type}**、每年行駛 **{annual_km:,}** 公里計算，"
        f"換購 {ev.model} {YEARS} 年累積{'可省下' if saved >= 0 else '反而多花'}約 **NT$ {abs(saved):,.0f}** 元的油資與保養總支出！"
    )
    summary_text += (
        f"\n\n📊 **資料依據（CSV）**：{maint_src_text(car_type)}，每年定保平均 NT$ {d['ice_maint_yr']:,.0f}；"
        f"{maint_src_text(ev.model)}，每年定保平均 NT$ {d['ev_maint_yr']:,.0f}。"
        f"能源費用為假設值：{d['km_per_l']:g} km/L × 汽油 NT$ {GAS_PRICE_PER_L:g}/L；"
        f"{ev.km_per_kwh:g} km/kWh × 平均電價 NT$ {plan['price']:.1f}/度（{'有固定車位，多數在家充' if home else '無固定車位，主要靠公共充電'}）。"
    )
    summary_text += "\n\n🔌 **充電需求推估**\n\n" + charge_text(plan, home) + "\n" + trip_text(trip_km, ev.model)
    return fig, summary_text

In [ ]:
# ---------------------------------------------------------
# 【區塊 5：建立 Gradio Web UI】
# 步驟 1 建立車主檔案 → 步驟 2 看 5 年 TCO → 步驟 3 與 AI 顧問對話
# ---------------------------------------------------------
GR_MAJOR = int(gr.__version__.split(".")[0])
BLOCKS_KW = {} if GR_MAJOR >= 6 else {"theme": gr.themes.Soft()}     # Gradio 6 起 theme 改放 launch()
LAUNCH_KW = {"theme": gr.themes.Soft()} if GR_MAJOR >= 6 else {}

PROFILE_CARS = CAR_CHOICES + [m for m in EV_CATALOG_ALL if m in MODEL_STATS.index and MODEL_STATS.loc[m, "owners"] > 0]   # 已經是電動車主也能建立檔案、問顧問；還沒車主資料的新車款不列入
EXAMPLE_QUESTIONS = ["大樓沒辦法裝充電樁怎麼辦？",
                     f"{DEFAULT_EV} 的保養費用和我現在的車比起來如何？",
                     f"幫我算出接下來 5 年的累積用車成本，油車和 {DEFAULT_EV} 比一比",
                     f"{DEFAULT_EV} 電池保固多久？",
                     "我每個月會開一次高速公路長途，電夠用嗎？"]
# 回覆完成後把游標放回輸入框；但使用者若已點去別的欄位（例如正在填檔案）就不搶焦點，也不捲動頁面
FOCUS_INPUT_JS = ("() => { setTimeout(() => { const t = document.querySelector('#msg_input textarea, #msg_input input');"
                  " const a = document.activeElement;"
                  " const free = !a || a === document.body || !!a.closest('#msg_input') || (a.tagName === 'BUTTON' && a.textContent.includes('送出'));"
                  " if (t && !t.disabled && free) t.focus({preventScroll: true}); }, 150); }")

def build_app():
    with gr.Blocks(title="TOYOTA SmartLife × EV LifePilot", **BLOCKS_KW) as demo:
        gr.Markdown("# 🚗 TOYOTA SmartLife × EV LifePilot - 個人化油轉電 AI 隨行顧問")
        gr.Markdown("先建立您的專屬檔案，AI 會為您精算 5 年持有成本，並以車主回廠資料回答您的購車與充電疑慮。")

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 📋 步驟 1：建立車主檔案")
                name_input = gr.Textbox(label="您的稱呼", placeholder="例如：魏先生")
                car_input = gr.Dropdown(PROFILE_CARS, label="目前駕駛車款", value=PROFILE_CARS[0])
                ev_input = gr.Dropdown(EV_MODELS, label="想比較的電動車款", value=DEFAULT_EV,
                                       visible=len(EV_MODELS) > 1)    # 目錄只有一台電動車時自動隱藏
                mileage_input = gr.Textbox(label="目前大約里程數", placeholder="例如：23000")
                km_input = gr.Slider(minimum=5000, maximum=50000, step=1000, label="預估年行駛里程 (公里)", value=15000)
                age_input = gr.Slider(minimum=0, maximum=20, step=1, label="目前車齡 (年)", value=5)
                visits_input = gr.Slider(minimum=0, maximum=12, step=1, label="近一年回廠次數", value=2)
                pref_input = gr.Textbox(label="您的用車習慣與喜好", placeholder="例如：每天通勤")
                home_input = gr.Checkbox(label="我家或公司有固定車位，可安裝充電樁", value=True)
                trip_input = gr.Slider(minimum=0, maximum=600, step=10, label="常跑的最長單趟距離 (公里，例如回南部、環島)", value=150)
                confirm_btn = gr.Button("✅ 確認並建立檔案", variant="primary")
                status_text = gr.Textbox(label="系統狀態", interactive=False)

            with gr.Column(scale=1):
                gr.Markdown("### 📊 步驟 2：5 年 TCO 養車成本精算")
                calc_btn = gr.Button("⚡ 立即精算 5 年 TCO 養車成本")
                output_text = gr.Markdown("建立檔案或點擊上方按鈕獲取診斷分析報告...")
                chart_output = gr.Plot(label="5 年持有成本比較圖")
                with gr.Accordion("🔎 業務端參考：客群診斷（驗證用，不會顯示在給客戶的對話裡）", open=False):
                    seg_output = gr.Markdown("建立檔案或點擊「立即精算」後顯示...")

        gr.Markdown("---")
        gr.Markdown("### 💬 步驟 3：24H 購車與充電疑慮諮詢 (GenAI 隨行顧問)")
        try:
            chatbot = gr.Chatbot(label="顧問回覆", height=500, type="messages")
        except TypeError:   # 新版 Gradio 已移除 type 參數，預設即 messages 格式
            chatbot = gr.Chatbot(label="顧問回覆", height=500)

        ready = gr.State(False)   # 車主檔案是否已建立
        busy = gr.State(False)    # 顧問是否正在回覆
        with gr.Row():
            msg_input = gr.Textbox(
                label="請輸入您的問題", placeholder=LOCKED_PLACEHOLDER,
                scale=4, interactive=False, elem_id="msg_input"
            )
            submit_btn = gr.Button("📤 送出", scale=1, interactive=False)

        gr.Markdown("試試看這些問法（點一下會填入輸入框）：")
        with gr.Row():
            for q in EXAMPLE_QUESTIONS:   # 只填文字，不碰輸入框的鎖定狀態（gr.Examples 會把它重設）
                gr.Button(q, size="sm").click(lambda q=q: q, None, msg_input, queue=False)

        # 建立檔案 → 順便跑一次 TCO 精算；之後改了車款/里程可再按「立即精算」
        confirm_btn.click(confirm_profile,
                          inputs=[name_input, car_input, mileage_input, km_input, pref_input],
                          outputs=[status_text, ready]
                          ).then(sync_chat_box, inputs=[ready, busy], outputs=[msg_input, submit_btn]
                          ).then(update_dashboard, inputs=[car_input, km_input, home_input, trip_input, ev_input], outputs=[chart_output, output_text]
                          ).then(segment_report, inputs=[car_input, age_input, visits_input, km_input, ev_input], outputs=seg_output)
        calc_btn.click(update_dashboard, inputs=[car_input, km_input, home_input, trip_input, ev_input], outputs=[chart_output, output_text]
                       ).then(segment_report, inputs=[car_input, age_input, visits_input, km_input, ev_input], outputs=seg_output)
        ev_input.change(update_dashboard, inputs=[car_input, km_input, home_input, trip_input, ev_input], outputs=[chart_output, output_text]
                        ).then(segment_report, inputs=[car_input, age_input, visits_input, km_input, ev_input], outputs=seg_output)   # 換比較車款就重算

        # 回覆期間輸入框與送出鈕會被鎖住（顯示「顧問思考中」），回覆完成後解鎖並把游標放回輸入框
        # Enter 與送出鈕合併成同一個事件(gr.on)，才不會在回覆期間被各送一次；concurrency_limit 讓多位使用者同時問時各自都看得到「思考中」
        chat_inputs = [msg_input, chatbot, name_input, car_input, mileage_input, km_input, age_input, visits_input, pref_input, home_input, trip_input, ready, ev_input]
        chat_outputs = [msg_input, chatbot, submit_btn, busy]
        gr.on(triggers=[msg_input.submit, submit_btn.click], fn=chat_with_ai,
              inputs=chat_inputs, outputs=chat_outputs, concurrency_limit=8
              ).then(sync_chat_box, inputs=[ready, busy], outputs=[msg_input, submit_btn]
              ).then(None, None, None, js=FOCUS_INPUT_JS)

    return demo

if __name__ == "__main__":
    demo = build_app()
    demo.launch(share=True, debug=True, **LAUNCH_KW)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8372a0c87efcef3394.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
